In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import models, transforms
import cv2
from sklearn.preprocessing import LabelEncoder
from torchvision import datasets
from torch.utils.data import DataLoader, ConcatDataset
import requests
import zipfile

In [ ]:
BACKEND_URL = "http://YOUR_NGROK_URL/api" 
API_KEY = "my_super_secret_key_123"

print(f"Connecting to Local Backend at {BACKEND_URL}...")
live_data_dir = './live_data'
headers = {"x-api-key": API_KEY}

In [ ]:
try:
    response = requests.get(f"{BACKEND_URL}/export-data", headers=headers)
    if response.status_code == 200:
        with open("verified_data.zip", "wb") as f:
            f.write(response.content)
        with zipfile.ZipFile("verified_data.zip", 'r') as zip_ref:
            zip_ref.extractall(live_data_dir)
        print("Successfully downloaded user-verified data from Local Backend!")
    else:
        print("No new verified data available on backend.")
except Exception as e:
    print(f"Could not connect to backend: {e}")

In [ ]:
class PlantDiseaseClassifier(nn.Module):
    def __init__(self, num_classes=38):
        super().__init__()
        self.model = models.efficientnet_b0(weights='DEFAULT')
        in_features = self.model.classifier[1].in_features
        self.model.classifier = nn.Identity()
        self.fc1 = nn.Linear(in_features, 512)
        self.relu1 = nn.ReLU()
        self.drop1 = nn.Dropout(0.4)
        self.fc2 = nn.Linear(512, 128)
        self.relu2 = nn.ReLU()
        self.drop2 = nn.Dropout(0.3)
        self.fc3 = nn.Linear(128, num_classes)